# Fine-tuning Qwen 2.5 1.5B on ConvoLearn Dataset

This notebook demonstrates:
1. Loading and splitting the dataset (1000 train, 250 test)
2. Fine-tuning Qwen 2.5 1.5B Instruct using LoRA
3. Evaluating both base and fine-tuned models
4. Plotting training loss across epochs

## 1. Install Required Libraries

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes peft trl torch tqdm matplotlib

## 2. Import Libraries

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Load and Split Dataset

In [ ]:
# Load the dataset
print("Loading dataset...")
ds = load_dataset("masharma/convolearn")
print(f"Dataset loaded: {ds}")

# Get the training split and select 1250 examples
full_dataset = ds['train'].select(range(1250))

# Split into train (1000) and test (250)
split_dataset = full_dataset.train_test_split(test_size=250, seed=42)
train_dataset = split_dataset['train']
test_dataset = split_dataset['test']

print(f"\nTrain dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"\nSample from dataset:")
print(train_dataset[0])

## 4. Configure Model and Tokenizer

In [ ]:
# Model configuration
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# BitsAndBytes configuration for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Qwen models have their own pad token, but set it if not available
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded successfully!")
print(f"Tokenizer vocab size: {len(tokenizer)}")

## 5. Load Base Model

In [ ]:
# Load base model with quantization
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Base model loaded successfully!")
print(f"Model device: {base_model.device}")

## 6. Prepare Data Formatting Function

In [ ]:
# Function to format the dataset for training with Qwen chat template
def format_instruction(sample):
    """
    Format the dataset examples into Qwen chat template format.
    Qwen uses a specific chat format with roles: system, user, assistant
    """
    # Check if the dataset has 'cleaned_conversation' field (ConvoLearn specific)
    if 'cleaned_conversation' in sample:
        conversation = sample['cleaned_conversation']
        
        # Parse the conversation format
        messages = []
        lines = conversation.split('\n')
        
        current_role = None
        current_content = []
        
        for line in lines:
            if line.startswith('Student:'):
                if current_role and current_content:
                    messages.append({
                        "role": current_role,
                        "content": ' '.join(current_content).strip()
                    })
                current_role = "user"
                current_content = [line.replace('Student:', '').strip()]
            elif line.startswith('Teacher:'):
                if current_role and current_content:
                    messages.append({
                        "role": current_role,
                        "content": ' '.join(current_content).strip()
                    })
                current_role = "assistant"
                current_content = [line.replace('Teacher:', '').strip()]
            elif line.strip() and current_role:
                current_content.append(line.strip())
        
        # Add the last message
        if current_role and current_content:
            messages.append({
                "role": current_role,
                "content": ' '.join(current_content).strip()
            })
        
        # Use Qwen's chat template
        if messages:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            return text
    
    # Fallback for other formats
    if 'conversations' in sample:
        messages = []
        for turn in sample['conversations']:
            role = turn.get('from', 'user')
            content = turn.get('value', '')
            if role in ['human', 'user']:
                messages.append({"role": "user", "content": content})
            elif role in ['gpt', 'assistant']:
                messages.append({"role": "assistant", "content": content})
        
        if messages:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            return text
    
    elif 'instruction' in sample and 'response' in sample:
        messages = [
            {"role": "user", "content": sample.get('instruction', '')},
            {"role": "assistant", "content": sample.get('response', '')}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return text
    
    # Default fallback
    return str(sample)

print("Data formatting function created!")
print("\nExample formatted output:")
formatted_sample = format_instruction(train_dataset[0])
print(formatted_sample[:500] + "..." if len(formatted_sample) > 500 else formatted_sample)

## 7. Configure LoRA

In [ ]:
# LoRA configuration
peft_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,  # LoRA alpha parameter
    lora_dropout=0.05,  # Dropout probability
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Target attention modules
)

# Prepare model for training
model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(model, peft_config)

# Print trainable parameters
model.print_trainable_parameters()

## 8. Custom Callback for Loss Tracking

In [ ]:
# Custom callback to track loss across epochs with enhanced progress tracking
class LossTrackingCallback(TrainerCallback):
    def __init__(self):
        self.losses = []
        self.epochs = []
        self.current_epoch = 0
        self.total_steps = 0
        self.current_step = 0
        
    def on_train_begin(self, args, state, control, **kwargs):
        self.total_steps = state.max_steps
        print(f"Training started: {args.num_train_epochs} epochs, {self.total_steps} total steps")
        
    def on_epoch_begin(self, args, state, control, **kwargs):
        self.current_epoch = int(state.epoch) if state.epoch else 0
        print(f"\n{'='*60}")
        print(f"Starting Epoch {self.current_epoch + 1}/{args.num_train_epochs}")
        print(f"{'='*60}")
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            self.current_step = state.global_step
            
            # Track loss
            if 'loss' in logs:
                self.losses.append(logs['loss'])
                self.epochs.append(state.epoch)
            
            # Print progress with more details
            progress_pct = (self.current_step / self.total_steps) * 100 if self.total_steps > 0 else 0
            
            log_str = f"Step {self.current_step}/{self.total_steps} ({progress_pct:.1f}%)"
            if 'loss' in logs:
                log_str += f" | Loss: {logs['loss']:.4f}"
            if 'learning_rate' in logs:
                log_str += f" | LR: {logs['learning_rate']:.2e}"
            
            print(log_str)
    
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_num = int(state.epoch) if state.epoch else 0
        if len(self.losses) > 0:
            recent_losses = [l for l, e in zip(self.losses, self.epochs) if int(e) == epoch_num]
            if recent_losses:
                avg_loss = sum(recent_losses) / len(recent_losses)
                print(f"\n{'='*60}")
                print(f"Epoch {epoch_num} completed | Average Loss: {avg_loss:.4f}")
                print(f"{'='*60}\n")
    
    def on_train_end(self, args, state, control, **kwargs):
        print(f"\n{'='*60}")
        print("Training completed!")
        print(f"Total steps: {self.current_step}")
        print(f"Final loss: {self.losses[-1]:.4f}" if self.losses else "No loss recorded")
        print(f"{'='*60}\n")

loss_callback = LossTrackingCallback()
print("Enhanced loss tracking callback created!")

## 9. Configure Training Arguments

In [ ]:
# from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-2.5-1.5b-finetuned",

    # training
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # optimization
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    weight_decay=0.001,
    max_grad_norm=0.3,

    # logging / saving - Enhanced for better progress tracking
    logging_steps=5,  # Log more frequently
    logging_first_step=True,
    save_strategy="epoch",
    report_to="none",
    disable_tqdm=False,  # Enable tqdm progress bars
    
    # precision - Changed to bf16 to fix BFloat16 CUDA error
    bf16=True,
    fp16=False,

    # NEW location for this
    packing=False,
)


print("Training arguments configured!")
print(f"Total epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Logging every {training_args.logging_steps} steps")

## 10. Initialize Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    formatting_func=format_instruction,
    callbacks=[loss_callback],
)

print("Trainer initialized successfully!")

## 11. Fine-tune the Model

In [ ]:
# Start training
print("Starting training...\n")
trainer.train()

print("\n" + "="*50)
print("Training completed successfully!")
print("="*50)

## 12. Save the Fine-tuned Model

In [ ]:
# Save the fine-tuned model
output_dir = "./qwen-2.5-1.5b-finetuned-final"
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Fine-tuned model saved to: {output_dir}")

## 13. Plot Training Loss

In [ ]:
# Plot loss across epochs
plt.figure(figsize=(12, 6))
plt.plot(loss_callback.epochs, loss_callback.losses, marker='o', linestyle='-', linewidth=2, markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.title('Training Loss Across Epochs - Qwen 2.5 1.5B Fine-tuning', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Loss plot saved as 'training_loss.png'")
print(f"\nFinal loss: {loss_callback.losses[-1]:.4f}")
print(f"Initial loss: {loss_callback.losses[0]:.4f}")
print(f"Loss reduction: {((loss_callback.losses[0] - loss_callback.losses[-1]) / loss_callback.losses[0] * 100):.2f}%")

## 14. Load Fine-tuned Model for Evaluation

In [ ]:
# Load the fine-tuned model
from peft import PeftModel

print("Loading fine-tuned model for evaluation...")
finetuned_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
finetuned_model = PeftModel.from_pretrained(finetuned_model, output_dir)
finetuned_model.eval()

print("Fine-tuned model loaded successfully!")

## 15. Define Evaluation Function

In [ ]:
def generate_response(model, tokenizer, prompt, max_length=256):
    """
    Generate response from the model given a prompt.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the response
    response = response[len(prompt):].strip()
    return response

def calculate_perplexity(model, tokenizer, text):
    """
    Calculate perplexity for a given text.
    """
    encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    encodings = {k: v.to(model.device) for k, v in encodings.items()}
    
    with torch.no_grad():
        outputs = model(**encodings, labels=encodings["input_ids"])
        loss = outputs.loss
    
    return torch.exp(loss).item()

print("Evaluation functions defined!")

## 16. Evaluate Base Model

In [ ]:
# Evaluate base model on test set
print("Evaluating base model...\n")
base_model.eval()

base_perplexities = []
base_responses = []
base_eval_data = []

# Evaluate on all 250 test samples
eval_samples = len(test_dataset)

for i in tqdm(range(eval_samples), desc="Base Model Evaluation"):
    sample = test_dataset[i]
    formatted_text = format_instruction(sample)
    
    # Calculate perplexity
    perplexity = calculate_perplexity(base_model, tokenizer, formatted_text)
    base_perplexities.append(perplexity)
    
    # Generate response for first few samples
    if i < 5:
        # Extract prompt (first part of formatted text)
        prompt = formatted_text.split('### Response:')[0] + '### Response:'
        response = generate_response(base_model, tokenizer, prompt)
        base_responses.append({
            'prompt': prompt,
            'response': response,
            'ground_truth': formatted_text
        })
    
    # Store data for CSV
    base_eval_data.append({
        'sample_id': i,
        'perplexity': perplexity,
        'formatted_text': formatted_text[:500]  # Truncate for CSV
    })

base_avg_perplexity = np.mean(base_perplexities)
print(f"\nBase Model Average Perplexity: {base_avg_perplexity:.2f}")
print(f"Base Model Perplexity Std: {np.std(base_perplexities):.2f}")

# Save to CSV
base_df = pd.DataFrame(base_eval_data)
base_df.to_csv('base_model_evaluation.csv', index=False)
print(f"\nBase model evaluation saved to 'base_model_evaluation.csv'")


## 17. Evaluate Fine-tuned Model

In [ ]:
# Evaluate fine-tuned model on test set
print("Evaluating fine-tuned model...\n")

finetuned_perplexities = []
finetuned_responses = []
finetuned_eval_data = []

for i in tqdm(range(eval_samples), desc="Fine-tuned Model Evaluation"):
    sample = test_dataset[i]
    formatted_text = format_instruction(sample)
    
    # Calculate perplexity
    perplexity = calculate_perplexity(finetuned_model, tokenizer, formatted_text)
    finetuned_perplexities.append(perplexity)
    
    # Generate response for first few samples
    if i < 5:
        prompt = formatted_text.split('### Response:')[0] + '### Response:'
        response = generate_response(finetuned_model, tokenizer, prompt)
        finetuned_responses.append({
            'prompt': prompt,
            'response': response,
            'ground_truth': formatted_text
        })
    
    # Store data for CSV
    finetuned_eval_data.append({
        'sample_id': i,
        'perplexity': perplexity,
        'formatted_text': formatted_text[:500]  # Truncate for CSV
    })

finetuned_avg_perplexity = np.mean(finetuned_perplexities)
print(f"\nFine-tuned Model Average Perplexity: {finetuned_avg_perplexity:.2f}")
print(f"Fine-tuned Model Perplexity Std: {np.std(finetuned_perplexities):.2f}")

# Save to CSV
finetuned_df = pd.DataFrame(finetuned_eval_data)
finetuned_df.to_csv('finetuned_model_evaluation.csv', index=False)
print(f"\nFine-tuned model evaluation saved to 'finetuned_model_evaluation.csv'")


## 18. Compare Results

In [ ]:
# Compare base and fine-tuned models
print("="*60)
print("EVALUATION RESULTS COMPARISON")
print("="*60)
print(f"\\nBase Model:")
print(f"  - Average Perplexity: {base_avg_perplexity:.2f}")
print(f"  - Perplexity Std: {np.std(base_perplexities):.2f}")
print(f"\\nFine-tuned Model:")
print(f"  - Average Perplexity: {finetuned_avg_perplexity:.2f}")
print(f"  - Perplexity Std: {np.std(finetuned_perplexities):.2f}")
print(f"\\nImprovement:")
improvement = ((base_avg_perplexity - finetuned_avg_perplexity) / base_avg_perplexity) * 100
print(f"  - Perplexity Reduction: {improvement:.2f}%")
print("="*60)

# Create comparison CSV
comparison_df = pd.DataFrame({
    'sample_id': range(eval_samples),
    'base_perplexity': base_perplexities,
    'finetuned_perplexity': finetuned_perplexities,
    'perplexity_improvement': [base_perplexities[i] - finetuned_perplexities[i] for i in range(eval_samples)]
})
comparison_df.to_csv('model_comparison.csv', index=False)
print(f"\\nComparison data saved to 'model_comparison.csv'")

## 19. Plot Perplexity Comparison

In [ ]:
# Plot perplexity comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
models = ['Base Model', 'Fine-tuned Model']
perplexities = [base_avg_perplexity, finetuned_avg_perplexity]
colors = ['#ff7f0e', '#2ca02c']

axes[0].bar(models, perplexities, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Average Perplexity', fontsize=12)
axes[0].set_title('Model Comparison - Average Perplexity', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add values on bars
for i, v in enumerate(perplexities):
    axes[0].text(i, v + 1, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

# Distribution plot
axes[1].hist(base_perplexities, bins=20, alpha=0.5, label='Base Model', color='#ff7f0e', edgecolor='black')
axes[1].hist(finetuned_perplexities, bins=20, alpha=0.5, label='Fine-tuned Model', color='#2ca02c', edgecolor='black')
axes[1].set_xlabel('Perplexity', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Perplexity Distribution', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('perplexity_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Perplexity comparison plot saved as 'perplexity_comparison.png'")

## 20. Sample Response Comparison

In [ ]:
# Display sample responses
print("="*80)
print("SAMPLE RESPONSE COMPARISON")
print("="*80)

for i in range(min(3, len(base_responses))):
    print(f"\n{'='*80}")
    print(f"SAMPLE {i+1}")
    print(f"{'='*80}")
    print(f"\nPROMPT:\n{base_responses[i]['prompt'][:200]}...")
    print(f"\n{'-'*80}")
    print(f"BASE MODEL RESPONSE:\n{base_responses[i]['response'][:300]}...")
    print(f"\n{'-'*80}")
    print(f"FINE-TUNED MODEL RESPONSE:\n{finetuned_responses[i]['response'][:300]}...")
    print(f"\n{'='*80}")

## 21. Summary Statistics

In [ ]:
# Final summary
print("\n" + "="*80)
print("FINE-TUNING SUMMARY")
print("="*80)
print(f"\\nDataset: masharma/convolearn")
print(f"Model: Qwen 2.5 1.5B Instruct")
print(f"Fine-tuning Method: LoRA (Supervised Fine-Tuning)")
print(f"\\nTraining:")
print(f"  - Training samples: {len(train_dataset)}")
print(f"  - Test samples: {len(test_dataset)}")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"\\nLoRA Configuration:")
print(f"  - Rank (r): {peft_config.r}")
print(f"  - Alpha: {peft_config.lora_alpha}")
print(f"  - Dropout: {peft_config.lora_dropout}")
print(f"\\nEvaluation Results:")
print(f"  - Base Model Perplexity: {base_avg_perplexity:.2f} ± {np.std(base_perplexities):.2f}")
print(f"  - Fine-tuned Model Perplexity: {finetuned_avg_perplexity:.2f} ± {np.std(finetuned_perplexities):.2f}")
print(f"  - Improvement: {improvement:.2f}%")
print(f"\\nModel saved to: {output_dir}")
print(f"\\nCSV Files Generated:")
print(f"  - base_model_evaluation.csv")
print(f"  - finetuned_model_evaluation.csv")
print(f"  - model_comparison.csv")
print("="*80)